# 04 Conference Revision Experiments

Purpose: add the conference-facing robustness checks requested in group feedback: structured interpretable baselines, fair RF vs engineered logistic comparison, non-selective evaluation setup, and random scenario-reduction diagnostics. Run this notebook after Notebook 01 outputs exist. Gurobi-dependent conference optimization and full-scenario evaluation are implemented in the final sections of Notebook 02.

## Phase 1: Structured baselines and risk matrices

This phase is Gurobi-free. It creates: `conference_baseline_model_comparison.csv`, `conference_risk_predictions_baselines_oof.csv`, `conference_scenario_node_probabilities_full_with_baselines.csv`, and `conference_optimization_risk_matrix_reduced_with_baselines.csv`.

In [ ]:
import os, math, json, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss, precision_recall_curve, f1_score


BASE = Path.cwd()
print("Current BASE:", BASE)
print("Files:", list(BASE.glob("storm_node_dataset_14nodes_200km_50kt.csv")))
OUT = BASE / 'outputs_04_conference_revision'
OUT.mkdir(exist_ok=True)
RANDOM_STATE = 42
N_SPLITS = 5

def ece_score(y_true, y_prob, n_bins=10):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (y_prob >= lo) & (y_prob < hi if hi < 1 else y_prob <= hi)
        if mask.sum() == 0:
            continue
        ece += (mask.sum() / len(y_true)) * abs(y_true[mask].mean() - y_prob[mask].mean())
    return float(ece)

def best_f1_threshold(y_true, y_prob):
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    f1s = 2 * precision * recall / np.maximum(precision + recall, 1e-12)
    idx = int(np.nanargmax(f1s))
    thr = thresholds[idx] if idx < len(thresholds) else 1.0
    return float(f1s[idx]), float(thr)

def metrics(y_true, y_prob):
    p = np.clip(np.asarray(y_prob, dtype=float), 1e-12, 1-1e-12)
    y = np.asarray(y_true)
    f1_best, thr_best = best_f1_threshold(y, p)
    return {
        'roc_auc': roc_auc_score(y, p),
        'average_precision': average_precision_score(y, p),
        'brier': brier_score_loss(y, p),
        'log_loss': log_loss(y, p),
        'ece_10bin': ece_score(y, p, n_bins=10),
        'f1_at_0p5': f1_score(y, p >= 0.5),
        'f1_best': f1_best,
        'best_threshold': thr_best,
    }

def safe_ohe():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)

# Load data
df = pd.read_csv(BASE / 'storm_node_dataset_14nodes_200km_50kt.csv')
# Merge extra final output columns where needed
final_rf = pd.read_csv(BASE / 'final_risk_predictions_oof_without_season.csv')
# Ensure stable ordering by SID,node_id
df = df.sort_values(['SID','node_id']).reset_index(drop=True)
final_rf = final_rf.sort_values(['SID','node_id']).reset_index(drop=True)
assert (df['SID'].values == final_rf['SID'].values).all() and (df['node_id'].values == final_rf['node_id'].values).all()

y = df['disrupted'].astype(int).values
groups = df['SID'].values

# Engineered features: no SEASON, no wind_at_closest or within-radius leakage features.
df_eng = df.copy()
df_eng['min_distance_sq'] = df_eng['min_distance_km'] ** 2
df_eng['max_wind_sq'] = df_eng['max_wind_kts'] ** 2
df_eng['mean_wind_sq'] = df_eng['mean_wind_kts'] ** 2
df_eng['distance_x_max_wind'] = df_eng['min_distance_km'] * df_eng['max_wind_kts']
df_eng['distance_x_mean_wind'] = df_eng['min_distance_km'] * df_eng['mean_wind_kts']
df_eng['coastal_x_max_wind'] = df_eng['coastal'] * df_eng['max_wind_kts']
df_eng['coastal_x_distance'] = df_eng['coastal'] * df_eng['min_distance_km']
df_eng['log_min_distance'] = np.log1p(df_eng['min_distance_km'])
df_eng['inv_min_distance'] = 1.0 / (1.0 + df_eng['min_distance_km'])

basic_numeric = ['min_distance_km', 'max_wind_kts', 'coastal']
base_numeric = [
    'month', 'duration_hours', 'max_wind_kts', 'mean_wind_kts',
    'mean_storm_speed', 'max_storm_speed', 'min_distance_km', 'coastal'
]
engineered_numeric = base_numeric + [
    # Explicit, physically interpretable nonlinear / interaction terms for a fair strong logistic baseline.
    'distance_x_max_wind',
    'distance_x_mean_wind',
    'coastal_x_max_wind'
]
cat_features = ['node_type']

def make_logit_pipeline(numeric_features, C=1.0):
    pre = ColumnTransformer([
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('ohe', safe_ohe())]), cat_features),
    ])
    clf = LogisticRegression(
        penalty='l2', C=C, class_weight='balanced', solver='liblinear',
        max_iter=2000, random_state=RANDOM_STATE
    )
    return Pipeline([('preprocess', pre), ('model', clf)])

models = {
    'Coastal/Inland': None,
    'Basic Logistic': make_logit_pipeline(basic_numeric, C=1.0),
    'Engineered Logistic': make_logit_pipeline(engineered_numeric, C=10.0),
    # RF re-run here for fair side-by-side with same folds; final RF from 01 also loaded below.
    'Random Forest rerun': Pipeline([
        ('preprocess', ColumnTransformer([
            ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), base_numeric),
            ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('ohe', safe_ohe())]), cat_features),
        ])),
        ('model', RandomForestClassifier(
            n_estimators=500, class_weight='balanced', min_samples_leaf=3,
            max_depth=None, random_state=RANDOM_STATE, n_jobs=1
        ))
    ]),
}

cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

def split_fit_calib(train_idx, random_state):
    train_y = y[train_idx]
    train_groups = groups[train_idx]
    splitter = GroupShuffleSplit(n_splits=100, test_size=0.25, random_state=random_state)
    best = None
    for k, (fit_rel, calib_rel) in enumerate(splitter.split(np.zeros(len(train_idx)), train_y, groups=train_groups)):
        fit_idx = train_idx[fit_rel]
        calib_idx = train_idx[calib_rel]
        if y[fit_idx].sum() >= 5 and y[calib_idx].sum() >= 3:
            return fit_idx, calib_idx
        best = (fit_idx, calib_idx)
    return best

def fit_sigmoid_calibrator(raw_prob, y_calib):
    # Map raw probabilities to calibrated probabilities via logistic regression on logit(p).
    p = np.clip(raw_prob, 1e-6, 1-1e-6)
    X = np.log(p/(1-p)).reshape(-1,1)
    cal = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=RANDOM_STATE)
    # fall back if calibration fold has one class
    if len(np.unique(y_calib)) < 2:
        return None
    cal.fit(X, y_calib)
    return cal

def apply_calibrator(cal, raw_prob):
    if cal is None:
        return raw_prob
    p = np.clip(raw_prob, 1e-6, 1-1e-6)
    X = np.log(p/(1-p)).reshape(-1,1)
    return cal.predict_proba(X)[:,1]

oof = pd.DataFrame({
    'SID': df['SID'], 'SEASON': df['SEASON'], 'NAME': df['NAME'], 'node_id': df['node_id'],
    'node_name': df['node_name'], 'node_type': df['node_type'],
    'min_distance_km': df['min_distance_km'], 'max_wind_kts': df['max_wind_kts'],
    'mean_wind_kts': df['mean_wind_kts'], 'coastal': df['coastal'], 'disrupted': y,
    'rf_final_p_uncalibrated': final_rf['p_uncalibrated'],
    'rf_final_p_calibrated': final_rf['p_calibrated'],
})

metric_rows = []
fold_rows = []

for model_name, pipe in models.items():
    raw_oof = np.zeros(len(df))
    cal_oof = np.zeros(len(df))
    for fold, (train_idx, test_idx) in enumerate(cv.split(df_eng, y, groups=groups), start=1):
        if model_name == 'Coastal/Inland':
            # Smoothed probabilities by coastal indicator, fitted on outer train only.
            train = df_eng.iloc[train_idx].copy()
            stats = train.groupby('coastal')['disrupted'].agg(['sum','count']).to_dict('index')
            global_p = (train['disrupted'].sum() + 1.0) / (len(train) + 2.0)
            preds = []
            for val in df_eng.iloc[test_idx]['coastal'].values:
                st = stats.get(val, {'sum':0, 'count':0})
                preds.append((st['sum'] + 1.0) / (st['count'] + 2.0) if st['count'] > 0 else global_p)
            raw = np.array(preds, dtype=float)
            cal = raw.copy()
        else:
            fit_idx, calib_idx = split_fit_calib(train_idx, RANDOM_STATE + fold)
            pipe.fit(df_eng.iloc[fit_idx], y[fit_idx])
            raw_calib = pipe.predict_proba(df_eng.iloc[calib_idx])[:,1]
            sigmoid = fit_sigmoid_calibrator(raw_calib, y[calib_idx])
            raw = pipe.predict_proba(df_eng.iloc[test_idx])[:,1]
            cal = apply_calibrator(sigmoid, raw)
        raw_oof[test_idx] = raw
        cal_oof[test_idx] = cal
        mr = metrics(y[test_idx], cal)
        mr.update({'model':model_name, 'fold':fold, 'n_test':len(test_idx), 'positives_test':int(y[test_idx].sum())})
        fold_rows.append(mr)
    oof[f'{model_name}_p_raw'] = raw_oof
    oof[f'{model_name}_p_calibrated'] = cal_oof
    for prob_type, prob in [('raw', raw_oof), ('calibrated', cal_oof)]:
        mr = metrics(y, prob)
        mr.update({'model':model_name, 'probability':'OOF_'+prob_type})
        metric_rows.append(mr)

# Final RF from 01 as benchmark.
for prob_type, col in [('raw','rf_final_p_uncalibrated'),('calibrated','rf_final_p_calibrated')]:
    mr = metrics(y, oof[col].values)
    mr.update({'model':'Random Forest final 01', 'probability':'OOF_'+prob_type})
    metric_rows.append(mr)

metrics_df = pd.DataFrame(metric_rows)
fold_df = pd.DataFrame(fold_rows)
metrics_df.to_csv(OUT / 'conference_baseline_model_comparison.csv', index=False)
metrics_df.to_csv(BASE / 'conference_baseline_model_comparison.csv', index=False)
fold_df.to_csv(OUT / 'conference_baseline_model_fold_results.csv', index=False)
oof.to_csv(OUT / 'conference_risk_predictions_baselines_oof.csv', index=False)
oof.to_csv(BASE / 'conference_risk_predictions_baselines_oof.csv', index=False)

# Fit full-data versions for scenario probability matrices.
# These are for optimization risk inputs, not for reporting predictive performance.
full_preds = pd.read_csv(BASE / 'final_scenario_node_probabilities_full_without_season.csv')
storm = df_eng.copy()
storm_key = storm.rename(columns={'SID':'scenario_id'})
merge_cols = ['scenario_id','node_id'] + engineered_numeric
full_merged = full_preds.merge(storm_key[merge_cols], on=['scenario_id','node_id'], how='left')

# Fit final models on all event-node data.
logit_full = make_logit_pipeline(engineered_numeric, C=10.0)
logit_full.fit(df_eng, y)
full_merged['p_engineered_logistic_raw'] = logit_full.predict_proba(full_merged)[:,1]
# Calibrate full logistic using OOF calibrated mapping? For risk matrix, use OOF-calibrated distribution by refitting sigmoid on OOF raw.
cal_logit = fit_sigmoid_calibrator(oof['Engineered Logistic_p_raw'].values, y)
full_merged['p_engineered_logistic_calibrated'] = apply_calibrator(cal_logit, full_merged['p_engineered_logistic_raw'].values)

# Coastal full smoothed.
stats = df_eng.groupby('coastal')['disrupted'].agg(['sum','count']).to_dict('index')
global_p = (df_eng['disrupted'].sum()+1.0)/(len(df_eng)+2.0)
def coastal_prob(v):
    st = stats.get(v, {'sum':0,'count':0})
    return (st['sum']+1.0)/(st['count']+2.0) if st['count']>0 else global_p
full_merged['p_coastal_inland'] = full_merged['coastal'].apply(coastal_prob)

# RF full final predictions already in full_preds columns.
full_out_cols = ['scenario_id','SEASON','NAME','node_id','node_name','node_type','disrupted','scenario_weight',
                 'p_is_uncalibrated','p_is_calibrated','p_coastal_inland',
                 'p_engineered_logistic_raw','p_engineered_logistic_calibrated']
full_out = full_merged[full_out_cols]
full_out.to_csv(OUT / 'conference_scenario_node_probabilities_full_with_baselines.csv', index=False)
full_out.to_csv(BASE / 'conference_scenario_node_probabilities_full_with_baselines.csv', index=False)

# Reduced/hybrid risk matrix with baselines.
hybrid = pd.read_csv(BASE / 'final_hybrid_reduced_hurricane_scenarios_without_season.csv')
reduced = hybrid[['scenario_id','scenario_weight']].drop_duplicates()
reduced_out = full_out.merge(reduced, on='scenario_id', how='inner', suffixes=('','_hybrid'))
# prefer hybrid weights
reduced_out['scenario_weight'] = reduced_out['scenario_weight_hybrid']
reduced_out = reduced_out.drop(columns=[c for c in ['scenario_weight_hybrid'] if c in reduced_out.columns])
reduced_out.to_csv(OUT / 'conference_optimization_risk_matrix_reduced_with_baselines.csv', index=False)
reduced_out.to_csv(BASE / 'conference_optimization_risk_matrix_reduced_with_baselines.csv', index=False)

# Risk representation summary.
rep_summary = []
for name, col in [
    ('RF calibrated','p_is_calibrated'),
    ('RF uncalibrated','p_is_uncalibrated'),
    ('Engineered Logistic calibrated','p_engineered_logistic_calibrated'),
    ('Engineered Logistic raw','p_engineered_logistic_raw'),
    ('Coastal/Inland','p_coastal_inland'),
    ('Rule binary','disrupted'),
]:
    vals = reduced_out[col].astype(float).values
    rep_summary.append({'risk_representation': name, 'mean': vals.mean(), 'std': vals.std(), 'min': vals.min(), 'max': vals.max(), 'nonzero_ratio': (vals>0).mean()})
summary_df = pd.DataFrame(rep_summary)
summary_df.to_csv(OUT / 'conference_risk_representation_summary_reduced.csv', index=False)
summary_df.to_csv(BASE / 'conference_risk_representation_summary_reduced.csv', index=False)

print('Saved outputs to', OUT)
print(metrics_df.sort_values(['probability','average_precision'], ascending=[True,False]).to_string(index=False))


## Phase 2: Scenario-reduction random baseline diagnostic

This checks whether the 29-scenario hybrid set is clearly different from random 29-scenario selections in terms of tail-risk retention. It does not claim full Pareto equivalence; it is a conference-version diagnostic against small-sample selection bias.

In [ ]:
# Phase 2: Scenario-reduction random baseline diagnostic
# Robust version for local Windows path and inconsistent old column names.

import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path.cwd()
OUT = BASE / "outputs_04_conference_revision"
OUT.mkdir(exist_ok=True)

# Load full scenario-node probability file
full_path = BASE / "final_scenario_node_probabilities_full_without_season.csv"
if not full_path.exists():
    raise FileNotFoundError(f"Missing file: {full_path}")

full = pd.read_csv(full_path)

# Robust column detection
risk_col_candidates = [
    "p_is_calibrated",
    "p_calibrated",
    "p_is",
    "p_rf_calibrated",
]
risk_col = next((c for c in risk_col_candidates if c in full.columns), None)
if risk_col is None:
    raise ValueError(f"Cannot find calibrated probability column. Columns are:\n{full.columns.tolist()}")

label_col_candidates = ["disrupted", "label", "exposed"]
label_col = next((c for c in label_col_candidates if c in full.columns), None)
if label_col is None:
    raise ValueError(f"Cannot find exposure label column. Columns are:\n{full.columns.tolist()}")

if "scenario_weight" not in full.columns:
    full["scenario_weight"] = 1.0 / full["scenario_id"].nunique()

group_cols = ["scenario_id"]
for c in ["SEASON", "NAME"]:
    if c in full.columns:
        group_cols.append(c)

agg = (
    full.groupby(group_cols, as_index=False)
    .agg(
        total_predicted_risk=(risk_col, "sum"),
        max_node_risk=(risk_col, "max"),
        observed_exposure_count=(label_col, "sum"),
        scenario_weight=("scenario_weight", "first"),
    )
)

all_ids = agg["scenario_id"].tolist()
all_weighted_total = float((agg["scenario_weight"] * agg["total_predicted_risk"]).sum())

def diagnostic_for_selected(selected_ids, name):
    selected_ids = set(selected_ids)
    sel = agg[agg["scenario_id"].isin(selected_ids)].copy()

    out = {
        "selection_method": name,
        "selected_count": len(sel),
        "scenario_weight_sum": float(sel["scenario_weight"].sum()),
        "mean_total_predicted_risk": float(sel["total_predicted_risk"].mean()),
        "max_total_predicted_risk": float(sel["total_predicted_risk"].max()),
        "mean_max_node_risk": float(sel["max_node_risk"].mean()),
        "max_node_risk": float(sel["max_node_risk"].max()),
        "weighted_total_risk_share": float(
            (sel["scenario_weight"] * sel["total_predicted_risk"]).sum() / all_weighted_total
        ),
    }

    for k in [5, 10, 20, 29]:
        for col, prefix in [
            ("total_predicted_risk", "total_predicted_risk"),
            ("max_node_risk", "max_predicted_risk"),
            ("observed_exposure_count", "observed_exposure_count"),
        ]:
            top = set(agg.sort_values(col, ascending=False).head(k)["scenario_id"])
            out[f"retention_top{k}_{prefix}"] = len(top & selected_ids) / k

    return out

# Recompute existing reduction-method diagnostics from selected scenario file if available
selected_path = BASE / "supp_selected_scenarios_by_reduction_method.csv"

existing_rows = []
if selected_path.exists():
    selected_df = pd.read_csv(selected_path)

    method_col = next(
        (c for c in ["selection_method", "reduction_method", "method"] if c in selected_df.columns),
        None
    )
    scenario_col = next(
        (c for c in ["scenario_id", "SID", "storm_id"] if c in selected_df.columns),
        None
    )

    if method_col is None or scenario_col is None:
        raise ValueError(
            "Cannot detect method/scenario columns in supp_selected_scenarios_by_reduction_method.csv. "
            f"Columns are:\n{selected_df.columns.tolist()}"
        )

    for method_name, temp in selected_df.groupby(method_col):
        existing_rows.append(diagnostic_for_selected(temp[scenario_col].unique(), method_name))

    existing = pd.DataFrame(existing_rows)
else:
    # Fallback: load old diagnostics, but it may not have all new columns
    old_diag_path = BASE / "supp_scenario_reduction_diagnostics.csv"
    if not old_diag_path.exists():
        raise FileNotFoundError(
            "Neither supp_selected_scenarios_by_reduction_method.csv nor "
            "supp_scenario_reduction_diagnostics.csv exists."
        )
    existing = pd.read_csv(old_diag_path)

# Random 29 scenario baseline
rows = []
for seed in range(1, 101):
    rng = np.random.default_rng(seed)
    selected = rng.choice(all_ids, size=29, replace=False)
    row = diagnostic_for_selected(selected, f"Random seed {seed}")
    row["seed"] = seed
    rows.append(row)

random_diag = pd.DataFrame(rows)

# Random summary
random_summary_rows = []
for col in [c for c in random_diag.columns if c not in ["selection_method", "seed"]]:
    vals = random_diag[col].dropna()
    if vals.dtype.kind in "if":
        random_summary_rows.append({
            "metric": col,
            "random_mean": vals.mean(),
            "random_std": vals.std(),
            "random_min": vals.min(),
            "random_p25": vals.quantile(0.25),
            "random_median": vals.median(),
            "random_p75": vals.quantile(0.75),
            "random_max": vals.max(),
        })

random_summary = pd.DataFrame(random_summary_rows)

# Add random mean row
rand_mean = {"selection_method": "Random 29 mean (100 seeds)"}
for c in random_diag.columns:
    if c not in ["selection_method", "seed"]:
        rand_mean[c] = random_diag[c].mean()

combined = pd.concat([existing, pd.DataFrame([rand_mean])], ignore_index=True)

# Save outputs
random_diag.to_csv(OUT / "conference_random_scenario_reduction_100seeds.csv", index=False)
random_summary.to_csv(OUT / "conference_random_scenario_reduction_summary.csv", index=False)
combined.to_csv(OUT / "conference_scenario_reduction_diagnostics_with_random.csv", index=False)
combined.to_csv(BASE / "conference_scenario_reduction_diagnostics_with_random.csv", index=False)

# Safe display
display_cols = [
    "selection_method",
    "selected_count",
    "weighted_total_risk_share",
    "retention_top5_total_predicted_risk",
    "retention_top10_total_predicted_risk",
    "retention_top20_total_predicted_risk",
    "retention_top10_observed_exposure_count",
]

display_cols = [c for c in display_cols if c in combined.columns]
display(combined[display_cols])

print("Saved:")
print(OUT / "conference_random_scenario_reduction_100seeds.csv")
print(OUT / "conference_random_scenario_reduction_summary.csv")
print(OUT / "conference_scenario_reduction_diagnostics_with_random.csv")

## Phase 3: Gurobi extension for conference multi-environment optimization

The corresponding Gurobi extension is implemented in the final conference sections of Notebook 02. Run Notebook 04 Phase 1 first to create the interpretable-baseline risk matrix, then run the conference sections of Notebook 02.